# CSC 349 HW5: Analysis

Answers based on the results from the diabetes notebook.


## 1. Implicit missing values

Implicit missing values can be hard to notice because they look like normal numbers instead of blank cells. In this dataset, zeros for glucose, blood pressure, skin thickness, insulin, and BMI are treated as missing because they are not meaningful measurements here. If the zeros are left in, they can change the averages and teach the model patterns based on values that are not actual measurements. A zero for pregnancies is possible, so I kept it.

I used the median to fill the missing measurements. The median is the middle value and is less affected by very large or small values than the mean. It is calculated from the training data and then used for both training and test data. This keeps the test set from influencing the preprocessing. Deleting every row with a missing measurement would remove many patients, especially because insulin has a lot of zeros. Keeping the rows lets the model use their other information. Imputation is still only an estimate and may hide differences between patients, so it is not the same as having the actual measurement.


## 2. Hard voting and soft voting

Hard voting asks each model for its predicted class and chooses the class with the most votes. For example, if two models predict diabetes and one predicts no diabetes, the final prediction is diabetes. Soft voting instead averages the probabilities that the models give for each class and picks the class with the higher average. This lets a model’s level of confidence affect the result.

Good calibration matters because a probability should roughly match how often the prediction is correct. If one model gives very confident probabilities even when it is often wrong, it can have too much influence on the average. This matters when combining different models because logistic regression, SVC, and a decision tree can produce probabilities in different ways. Setting probability=True allows the SVC to give probabilities, but it does not guarantee that every model is well calibrated. Calibration could be checked using validation data.


## 3. Out-of-bag evaluation

In bagging, each tree trains on a bootstrap sample drawn from the training data with replacement. Because some rows are picked more than once, other training rows are left out of that tree’s sample. Those left-out rows are called out-of-bag rows. Each row can be predicted using the trees that did not train on it, and those predictions are used to calculate the OOB score.

The OOB score is useful because it gives an internal performance estimate while the ensemble is being trained. It does not require training separate sets of models for multiple cross-validation splits, so it can save time. My OOB score was 0.756 and the held-out test accuracy was 0.721. They are separate estimates and do not have to match. The imputer and scaler were fitted on all training rows before the trees were bootstrapped, so the OOB score is an approximation and is not as isolated as a full cross-validation pipeline. The test set was kept separate.


## 4. Random forests and feature subsetting

Both bagging and random forests train trees using bootstrap samples of the training rows. In standard bagging with decision trees, a tree can consider all features when choosing each split. In a random forest, each split considers only a random subset of features. In this notebook I used max_features='sqrt'.

This matters because a strong feature such as glucose might otherwise be chosen by many trees, making them more alike. Random feature choices give other measurements a chance to be used and can make the trees less similar. When their predictions are combined, the differences between trees can reduce variance and help the model work better on new data. This is not guaranteed for every dataset, and feature importance shows how the model used a feature, not that the feature caused diabetes.


## 5. AdaBoost and overfitting

AdaBoost trains one decision stump at a time. Each stump has one split. After a stump makes predictions, AdaBoost gives more weight to the rows it got wrong so the next stump pays more attention to those cases. The final prediction combines the stumps, with their contributions based on how well they performed.

My training accuracy was 0.814 and my test accuracy was 0.760. The training score was higher by 0.055. This gap suggests some overfitting, since the model did better on the data it learned from, but one split alone cannot prove how well it will generalize. To try to reduce overfitting, I could lower the learning rate or try fewer stumps. The number of stumps and learning rate work together, so I would compare combinations using cross-validation on the training set and then check the test set after choosing the settings. More stumps or a high learning rate can also make the model pay too much attention to noisy or mislabeled rows.
